# Hyperforge AI - Colab Pro Deployment (Image-Only, Uncensored)
This notebook launches the FLUX.1-dev Image Generation server directly on Google Colab.
It uses a **Cloudflare Tunnel** to provide a public URL to access the UI without requiring any accounts or manual setup.

In [ ]:
# 1. Setup repository and install dependencies

# Clone your public repository (No authentication required!)
!git clone https://github.com/krishnagopalmishra1-we/flux-server.git /content/hyperforge || echo 'Repo already exists'
%cd /content/hyperforge/flux-server

# Install required dependencies
!pip install -r requirements.txt
!pip install fastapi uvicorn

In [ ]:
# 2. Setup Cloudflare Tunnel (No account needed)
!wget -q -c -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x cloudflared-linux-amd64
!nohup ./cloudflared-linux-amd64 tunnel --url http://localhost:8080 > /content/cloudflared.log 2>&1 &

In [ ]:
# 3. Start the FastAPI server in the background
import os
try:
    from google.colab import userdata
    # Make sure you have added a secret named 'HF_TOKEN' in the Colab Secrets tab (key icon on the left)
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
    print("Successfully loaded HF_TOKEN from Colab Secrets.")
except Exception as e:
    print("WARNING: HF_TOKEN not found in Colab Secrets! Models may fail to download.")

!nohup uvicorn app.main:app --host 0.0.0.0 --port 8080 > /content/server.log 2>&1 &

In [ ]:
# 4. Get the Public URL
import time
import re
print("Waiting for Cloudflare tunnel to establish and Server to boot...")
time.sleep(10)

with open("/content/cloudflared.log", "r") as f:
    log_text = f.read()
    url_match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", log_text)
    if url_match:
        print("\n\033[92m" + "="*70)
        print("SUCCESS! Your Uncensored Image Studio is running.")
        print("Click this link to open the UI: \033[94m\033[1m" + url_match.group(0) + "\033[0m")
        print("\033[92m" + "="*70 + "\033[0m\n")
    else:
        print("Could not find the URL in the logs. Checking logs:")
        print(log_text)

print("\nServer logs (running in background):")
!tail -n 15 /content/server.log